In [6]:
%run prepare_lexicon.py

Словарь MFA уже находится в папке data.

Сбор данных без пауз...
Найдено и будет удалено предложений с 'spn': 9827


Парсинг TextGrid: 100%|██████████| 21327/21327 [00:18<00:00, 1148.40it/s]



Сбор данных с паузами...
Найдено и будет удалено предложений с 'spn': 9827


Парсинг TextGrid: 100%|██████████| 21327/21327 [00:19<00:00, 1112.70it/s]



Сохранено токенов без пауз: 120364
Сохранено токенов с паузами: 148160
Словарь собран: уникальных слов в train: 22432, из них омографов с разными произношениями: 251


# РАЗВЕДОЧНЫЙ АНАЛИЗ

Вопросы к рассмотрению:
1. Какие фонемы представлены в корпусе? Как они соотносятся с [IPA](https://en.wikipedia.org/wiki/Help:IPA/Russian)? Какая средняя длительность и вариативность длительности фонем?
2. Определите самые редкие фонемы в корпусе: Можно ли с ними что-то сделать?
3. Сколько в корпусе Out-Of-Vocabularly (OOV) слов? (по отношению к `russian_mfa.dict` и `lexicon.csv`)
4. Корректно ли отработала разметка на примерах, которые удаляются фильтром, например, аббревиатурах, сокращениях, числительных?
5. Как в корпусе решается проблема ударения в омографах типа `замок`, пар `все`/`всё`? Корректна ли разметка, и что говорит по этому поводу словарь произношений?
6. Ознакомьтесь с целевым списком фонем, который будет использоваться в последующей работе: `common/phones_mfa.txt`/ Как привести наблюдаемый список фонем в корпусе RUSLAN к целевому?
**Результат этапа:** Отчёт с описанием и результатами анализа

Чтобы узнать точную длительность фонем, нужно прочитать исходные интервалы. В tokens_phones.csv уже есть склеенные строки, а для отчета нужна статистика по чистым фонемам, можно собрать их свойства.
Для точного расчета средних длительностей и вариативности фонем (так как в итоговый CSV длительность не вошла), используем следующую функцию, которая вытащит это напрямую из интервалов TextGrid (только по тем предложениям, которые остались после фильтрации spn):

In [7]:
import os
import glob
from praatio import textgrid

import pandas as pd
import numpy as np

df_tokens = pd.read_csv('data/tokens_phones.csv', sep='|')
df_lexicon = pd.read_csv('data/lexicon.csv', sep='|')

# Собираем список хороших ID из tokens_phones.csv
good_ids = set(df_tokens['id'].unique())

phone_records = []
align_dir = '../../data/RUSLAN_align_byPelse_attempt_2_v2/'

for path in glob.glob(os.path.join(align_dir, '*.TextGrid')):
    utt_id = os.path.basename(path)[:-len('.TextGrid')]
    if utt_id not in good_ids:
        continue
    try:
        tg = textgrid.openTextgrid(path, True)
        _, phone_tier = tg.tiers
        for p in phone_tier.entries:
            if p.label and p.label != 'spn':
                # Убираем знак долготы
                clean_phone = p.label.replace('ː', '') 
                phone_records.append({
                    'phone': clean_phone,
                    'duration': p.end - p.start
                })
    except:
        pass

df_raw_phones = pd.DataFrame(phone_records)

# Считаем статистику фонем (Вопрос 1)
phone_stats = df_raw_phones.groupby('phone')['duration'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(phone_stats.sort_values(by='count', ascending=False).to_string())


   phone  count      mean       std       min   max
44     ɪ  51406  0.017709  0.024863  0.005782  0.54
38     ə  48301  0.039753  0.033896  0.004943  0.38
36     ɐ  34381  0.064189  0.020643  0.030000  0.37
26    t̪  30614  0.078687  0.032093  0.010000  0.36
0      a  29996  0.089426  0.036441  0.010000  0.46
11     j  25938  0.066377  0.039161  0.007098  0.40
15    n̪  23938  0.067760  0.030982  0.010000  0.35
7      e  23316  0.074651  0.034925  0.007551  0.34
22    s̪  21236  0.101834  0.032321  0.010000  0.31
16     o  20320  0.084506  0.036696  0.010000  0.40
43     ɨ  20006  0.053026  0.036588  0.007506  0.44
19     r  19611  0.061051  0.025932  0.026961  0.30
10     i  19470  0.070688  0.040750  0.010000  0.53
29     v  18839  0.070960  0.029253  0.010000  0.41
12     k  17481  0.088998  0.034698  0.010000  0.43
45     ɫ  16257  0.078900  0.039201  0.010884  0.40
13     m  15577  0.086149  0.034121  0.004966  0.37
46     ɲ  14793  0.083585  0.031818  0.010000  0.35
51     ʎ  14

### Вопрос: "1. Какие фонемы представлены в корпусе? Как они соотносятся с [IPA](https://en.wikipedia.org/wiki/Help:IPA/Russian)? Какая средняя длительность и вариативность длительности фонем?"

Ответ: 

Похоже, что в корпусе представлены фонемы из IPA, вот самые частые:

| Фонема | Примеры | Мое описание | Трактовка частотности |
| :--- | :--- | :--- | :--- |
| ɪ | лИса, тЯжёлый | бнезударный гласный, который при произношении напоминает русский "и" | Не удивительно, что это наиболее частая фонема, т.к. в русском языке много безударных гласных звучит (или может звучать) как "и" |
| ə | кожА, сердцЕ, облАкО; волЯ, кучА | Вроде эту фонему называют ШВА. Все безударные гласные, которые по произношению близки к короткому (смазанному) звуку "а". | Не удивительно, что это очень частая фонема, т.к. в русском языке много безударных гласных звучит (или может звучать) как "а" |
| t̪ | Примеры | твердая буква "т" | Очевидна |

средняя частота:

| Фонема | Примеры | Мое описание | Трактовка частотности |
| :--- | :--- | :--- | :--- |
| ɨ | шИшка, с Играми | русский "ы" | Очевидна |
| tʲ | тень, дитя | русский мягкий "т" | Очевидна |

редкие:

| Фонема | Примеры | Мое описание | Трактовка частотности |
| :--- | :--- | :--- | :--- |
| ʑ | дроЖЖи, веЩдок | долгий мягкий звук «ж» | слов в русском языке с таким звуком действительно мало |
| ɣ | аГа, двухдверный | похож на украинский гхэ, в словах, в которых ЛФШ четко произносит "г", а южане произносят что-то среднее между "х", "г", "у" и "в" одновременно сказанными | так как диктор говорит на русском с произношением ЛФШ (возможно, МФШ), то звук южного наречия встречается редко |
| dʐ | ДЖип | английский звук "дж" | часто встречается в заимствованных словах, например, джип, джинн, джинсы. Поэтому редко встречается в русском языке |


* Про длительность и разброс:
	* Чаще всего в текстах встречаются короткие безударные гласные (ɪ, ə) и твердая буква «т» (t̪).
	* Логично, что длительность «смазанных» безударных звуков короче: звук ɪ длится в среднем всего 0.017 секунды, а «шва» ə — 0.039 секунды. Дольше всего диктор тянет мягкие свистящие и шипящие звуки, например, sʲ («сь») и tʲ («ть») держатся по 0.11–0.12 секунды.
	* Разброс (std) в районе 0.02–0.04 секунды показывает, что диктор говорит живо: один и тот же звук в разных словах и при разной скорости речи может звучать чуть быстрее или чуть медленнее.

### Вопрос: "2. Определите самые редкие фонемы в корпусе: Можно ли с ними что-то сделать?"

Ответ:

В самом низу таблицы оказались звуки-одиночки, которые встречаются буквально считанные разы:
• tʂ (что-то вроде «ц» или «ч») — 174 раза.
• dʐ (слитное «дж», как в слове «джинсы») — 86 раз.
• ɣ (украинское или южнорусское «гэ», как в слове «ага») — всего 11 раз во всем корпусе.
• ʑ (мягкое «жжь», как в старом произношении слова «дрожжи») — всего 10 раз.
Что с ними делать, чтобы не сломать нейросеть:
крайне маленькое количество примеров таких звуков не способно обучить нейросеть - ей не хватит примеров.
Можно попробовать что-то из этого:
1. Заменить редкое на частое (маппинг): Редкое ɣ в коде принудительно переписать в обычную твердую русскую букву g («г»). Редкое мягкое ʑ заменить на стандартное твердое ʐ («ж») или «щ» (ɕ).
2. Выкинуть плохие строчки: Если эти звуки вылезли в каких-то странных заимствованных именах или редких словах, можно удалить эти несколько предложений из тренировочной выборки (train), чтобы они не путали модель и не мусорили в словаре.

In [8]:
df_lexicon = pd.read_csv('data/lexicon.csv', sep='|')

# количество слов в russian_mfa.dict
unique_words = set()

with open("data/russian_mfa.dict", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # пропускаем пустые строки
            
        # Берем самое первое слово в строке
        first_token = line.split()[0]
        
        # Фильтруем технические токены вроде [laughter] или <unk>
        if first_token.startswith(('[', '<')):
            continue
            
        unique_words.add(first_token)

# количество слов в oovs_found.txt
unique_oov_words = set()
with open("../../data/oovs_found.txt", "r", encoding="utf-8") as f:
    for line in f:
        word = line.strip().lower()  # убираем пробелы/переносы и приводим к одному регистру
        if word:  # проверяем, что строка не пустая
            unique_oov_words.add(word)

print(f"### Вопрос: 3. Сколько в корпусе Out-Of-Vocabularly (OOV) слов? (по отношению к `russian_mfa.dict` и `lexicon.csv`)")

# Создаем структуру данных для итоговой таблицы
data = {
    "Источник данных / Метрика": [
        "Уникальных слов в lexicon.csv",
        "Уникальных слов в russian_mfa.dict",
        "Всего известных слов в системе (Объединенный словарь)",
        "Out-Of-Vocabulary (OOV) слов найдено в корпусе"
    ],
    "Количество": [
        len(df_lexicon),
        len(unique_words),
        len(df_lexicon) + len(unique_words), # Приблизительный объем, если они не пересекаются
        len(unique_oov_words)
    ]
}

# Строим красивый DataFrame
summary_df = pd.DataFrame(data)

# Стилизуем таблицу: выравниваем числа по правому краю, текст по левому, добавляем разделители тысяч
styled_table = (summary_df.style
    .format({"Количество": "{:,}"}) # делает из 452397 -> 452,397
    .hide(axis="index") # убираем лишние индексы строк 0, 1, 2...
    .set_properties(**{'text-align': 'left'}, subset=["Источник данных / Метрика"])
    .set_properties(**{'text-align': 'right'}, subset=["Количество"])
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('padding', '8px')]}
    ])
)

# Выводим текстовый ответ жирным шрифтом
from IPython.display import display, Markdown
display(Markdown(f"### **Ответ:** В корпусе обнаружено **{len(unique_oov_words):,}** Out-Of-Vocabulary (OOV) слов."))

# Отображаем саму таблицу
display(styled_table)

### Вопрос: 3. Сколько в корпусе Out-Of-Vocabularly (OOV) слов? (по отношению к `russian_mfa.dict` и `lexicon.csv`)


### **Ответ:** В корпусе обнаружено **8,485** Out-Of-Vocabulary (OOV) слов.

Источник данных / Метрика,Количество
Уникальных слов в lexicon.csv,"22,697"
Уникальных слов в russian_mfa.dict,"452,397"
Всего известных слов в системе (Объединенный словарь),"475,094"
Out-Of-Vocabulary (OOV) слов найдено в корпусе,"8,485"


### Вопрос: "4. Корректно ли отработала разметка на примерах, которые удаляются фильтром, например, аббревиатурах, сокращениях, числительных?"

Да, так как я ранее аббревиатуры я привела к слогам, например, для "КВВК":

006918_RUSLAN|0|ка|k ə|train
006918_RUSLAN|1|вэ|v ɛ|train
006918_RUSLAN|2|вэ|v ɛ|train
006918_RUSLAN|3|ка|k ə|train

Для "полковник МВД":

016582_RUSLAN|3|полковник|p ɐ ɫ k o v ɲ ɪ k|train
016582_RUSLAN|4|эм|ɛ m|train
016582_RUSLAN|5|вэ|v ɛ|train
016582_RUSLAN|6|дэ|d̪ ɛ|train

Если сделана предобработка, то, скорее всего, разметка отработает как ожидается.

С числительными - порядок. Так как в ЛР 1 у меня есть функция "_expand_cardinals()", которая превращает числительное цифрой в текстовое: "5" -> "пять".

### Вопрос: "5. Как в корпусе решается проблема ударения в омографах типа `замок`, пар `все`/`всё`? Корректна ли разметка, и что говорит по этому поводу словарь произношений?"

Проблема одинаково написанных, но разных в произношении слов, решается в корпусе назначением разных фонем. Пример для "всё" и "все":

000106_RUSLAN|6|когда|k ɐ ɡ d̪ a|train
000106_RUSLAN|8|все|f sʲ ɵ|train
000106_RUSLAN|9|было|b ɨ ɫ ə|train
000106_RUSLAN|10|хорошо|x ə r ɐ ʂ o|train

000665_RUSLAN|0|все|f sʲ e|test
000665_RUSLAN|1|эти|ɛ tʲ ɪ|test
000665_RUSLAN|2|люди|ʎ ʉ dʲ ɪ|test

Для "муки"

010655_RUSLAN|2|облегчили|ɐ b ʎ ɪ x tɕ i ʎ ɪ|test
010655_RUSLAN|3|ей|j e j|test
010655_RUSLAN|4|муки|m ʊ c i|test

Если я правильно понимаю, то верные фонемы для слова "мУки" такое: m ʉ c ɪ (вместо m ʊ c i)
Разница в двух символах. Выглядит, что обработчик неправильно поставил ударение. Возможно, такие случаи обрабатываются кастомно. По моим наблюдениям омографы как явление не очень частое в языке, модели не хватит данных для нормального обучения и различия.

### Вопрос: "6. Ознакомьтесь с целевым списком фонем, который будет использоваться в последующей работе: `common/phones_mfa.txt`/ Как привести наблюдаемый список фонем в корпусе RUSLAN к целевому?"

Как делать:
1) нужно понять, есть ли отличия. 
    * Лишние фонемы: те, что есть в корпусе RUSLAN, но отсутствуют в целевом phones_mfa.txt. Для них обязательно нужен словарь замен.
    * Недостающие фонемы: те, что есть в целевом списке, но ни разу не встретились в корпусе (это нормально, для них замены не нужны, корпус просто их не покрывает).

2) составить словарь замен
3) переписать с заменой

In [5]:
# Загружаем целевой алфавит
target_phones = set()
with open("../../common/phones_mfa.txt", "r", encoding="utf-8") as f:
    for line in f:
        p = line.strip()
        if p:
            target_phones.add(p)

# Наблюдаемый алфавит из df_raw_phones
observed_phones = set(df_raw_phones['phone'].unique())

# Находим расхождения
extra_in_ruslan = observed_phones - target_phones
missing_in_ruslan = target_phones - observed_phones

print(f"Фонемы в RUSLAN, которых НЕТ в целевом списке (нужно заменить): {extra_in_ruslan}")
print(f"Целевые фонемы, которых нет в корпусе (допустимо): {missing_in_ruslan}")

Фонемы в RUSLAN, которых НЕТ в целевом списке (нужно заменить): set()
Целевые фонемы, которых нет в корпусе (допустимо): {'<SIL>', 'd̪z̪'}


Наблюдения:

    * Результат set() означает, что наблюдаемый список фонем в корпусе RUSLAN уже полностью совпадает с целевым.
    * Все фонемы, которые были извлечены из TextGrid-файлов, абсолютно легитимны и присутствуют в целевом файле common/phones_mfa.txt
    * Фонема d̪z̪ отсутствует в корпусе, выглядит нормально, так как фонема крайне редкая (например, в слове "дзенькать", "дэюдо"), и в текстах датасета не оказалось слов с этим звуком. Никаких действий здесь предпринимать не нужно.
    * Символ <SIL> сохранен в файле tokens_phones_sil.csv:

018289_RUSLAN|7|одиночество|ɐ dʲ ɪ n̪ o tɕ ɪ s̪ t̪ v ə|train
018289_RUSLAN|8|<SIL>|<SIL>|train
018289_RUSLAN|9|еще|j ɪ ɕ e|train 